# Data Cleaning & EDA — Vehicle Theft in Argentina (2022–2023)

Portfolio reconstruction of the original **Laboratorio de Datos** project.

**Pipeline:** DNRPA → Power Query → R / EDA → SQLite / SQL → Power BI.

The final workbook contains **83,840 records and 34 columns**. This notebook runs on a small deterministic sample so the repository remains lightweight; full-data metrics in the narrative were recalculated from the complete workbook.

In [ ]:
from pathlib import Path
import math, sqlite3
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import chi2_contingency

df_raw = pd.read_csv(Path("../data/robo_autos_sample.csv"))
print(df_raw.shape)
df_raw.head()

## 1. Data quality and cleaning

The original work standardized dates, provinces, vehicle types, makes and models; created derived variables; and audited missing or suspicious values.

A fresh audit of the complete final workbook found **23,079 empty cells**, concentrated mainly in `Tipo_descripcion2` (11,397), country IDs (4,750), vehicle-type codes (2,399) and model identifiers.

The original rule for suspicious holder birth years was **≤1934 or ≥2005**. Recalculation flags **1,588 records (1.89%)**. In this portfolio version they are flagged for review rather than automatically deleted or imputed.

In [ ]:
df = df_raw.copy()
for col in ["tramite_fecha","fecha_inscripcion_inicial","tramite_fecha2"]:
    df[col] = pd.to_datetime(df[col], errors="coerce")
for col in ["automotor_anio_modelo","titular_anio_nacimiento","Año_Robo"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df["birth_year_flag"] = (
    df["titular_anio_nacimiento"].notna()
    & ((df["titular_anio_nacimiento"] <= 1934) | (df["titular_anio_nacimiento"] >= 2005))
)

missing = df.isna().sum().sort_values(ascending=False)
missing.head(10).to_frame("missing_n")

## 2. Exploratory analysis

These are administrative records, not a probability sample. Counts describe concentration **within the dataset**, not population incidence rates.

Full-workbook benchmarks:
- Buenos Aires: **55,848** records; CABA: **10,439**; Córdoba: **7,021**; Santa Fe: **4,048**.
- Leading cleaned makes: Volkswagen **17,350**, Chevrolet **11,534**, Fiat **11,361**, Renault **11,221**.
- Leading make/model: Volkswagen Gol **8,366**.
- Among records coded male or female, the split is **66.7% / 33.3%**.

In [ ]:
top_makes = df["automotor_marca_limpio"].value_counts().head(10).sort_values()
top_makes.plot(kind="barh", figsize=(7,4), title="Most frequent makes — sample")
plt.xlabel("Records"); plt.ylabel(""); plt.tight_layout(); plt.show()

In [ ]:
top_provinces = df["registro_seccional_provincia"].value_counts().head(10).sort_values()
top_provinces.plot(kind="barh", figsize=(7,4), title="Records by province — sample")
plt.xlabel("Records"); plt.ylabel(""); plt.tight_layout(); plt.show()

## 3. Categorical association

The original R analysis tested vehicle origin × holder gender. Recalculation on the complete workbook gives **χ²≈357.60, df=6, p<.001**. With more than 80k records, statistical significance alone can exaggerate substantive importance; **Cramér's V≈0.046**, indicating a very small association.

In [ ]:
tab = pd.crosstab(df["automotor_origen"], df["titular_genero"])
chi2, p, dof, _ = chi2_contingency(tab)
n = tab.to_numpy().sum()
v = math.sqrt(chi2 / (n * min(tab.shape[0]-1, tab.shape[1]-1)))
print(f"Sample chi-square={chi2:.2f}, df={dof}, p={p:.4g}, Cramér's V={v:.3f}")
tab

## 4. SQL layer

The processed data were stored in SQLite using DB Browser. The repository preserves the recovered SQL and adds a reorganized query file.

In [ ]:
with sqlite3.connect(":memory:") as con:
    df.to_sql("dnrpa", con, index=False, if_exists="replace")
    q = '''
    SELECT automotor_marca_limpio AS make, COUNT(*) AS records
    FROM dnrpa
    GROUP BY automotor_marca_limpio
    ORDER BY records DESC
    LIMIT 10
    '''
    display(pd.read_sql_query(q, con))

## 5. What this project demonstrates

**Data preparation:** integration, type correction, text standardization and derived variables.  
**Data quality:** missingness audit, plausibility checks and explicit treatment of suspicious values.  
**EDA:** frequencies, geographic concentration, model-year patterns and contingency analysis.  
**Data engineering / BI:** SQLite, SQL and Power BI.

The strongest portfolio value is the traceable path from raw administrative data to cleaned analytical variables and reproducible summaries.